In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import cm

ad.settings.allow_write_nullable_strings = True
sc.settings.verbosity = 0


In [ ]:
def load_evidence(h5ad_path):
    print('Loading Evidence from the previous Projected dataset split B')
    adata = sc.read_h5ad(h5ad_path)
    print(f'Loaded B set matrix of dimensions \
          {adata.n_obs} cells and {adata.n_vars} genes')
    return adata
    

In [ ]:
def wilcoxon_test(adata):
    sc.tl.rank_genes_groups(adata,groupby='leiden',method='wilcoxon',tie_correct=True,
                            layer='log1p_norm')
    return adata


In [ ]:
# 1. The Data: A dictionary of 4 different signal arrays
def pval_hist(adata):
# 2. Divide the Space: 2 rows, 2 columns (Total 4 Axes)
    a = adata.uns['rank_genes_groups']['pvals']
    fig, axes = plt.subplots(nrows=4, ncols=2, figsize=(50, 40))

    # 3. Transform the 2x2 matrix into a 1D list of 4 objects for easy looping

    axes_flat = axes.flatten()

    # 4. The Iteration 
    for i in a.dtype.names:
        
        # Select the specific coordinate plane
        current_ax = axes_flat[int (i)]
        values = a[i]
        # Paint the data (Using Matplotlib hist)
        current_ax.hist(values, bins=20, color='blue', edgecolor='green')
        
        # Enforce metadata per plane
        current_ax.set_title(f"Cluster: {i}")
    # 5. Prevent geometric overlapping of labels
    plt.tight_layout()
    plt.savefig('/Users/qgem/GitHub/PBMC3k-reproducible/notebooks/figures/pvals.png')
    plt.close()

In [ ]:
def volcano_logfc_pvals_adj(adata):
    c = adata.uns['rank_genes_groups']['pvals_adj']
    d = adata.uns['rank_genes_groups']['logfoldchanges']
    fig, axes = plt.subplots(nrows=4, ncols=2, figsize=(15, 12))
    axes_flat = axes.flatten()
    for i in c.dtype.names:
        
        current_ax = axes_flat[int (i)]
        neg_log10_padj = -np.log10(c[i]+ 1e-300)
        logFC_val = d[i]
        fdr_cut = 2.70
        logFC_cut = 2.0
        
        
        
        # SATURN GUARDRAIL:
        # If the 20th gene is boring (LogFC < 0.5), force a minimum floor of 1.0
        # This prevents highlighting noise in weak clusters.
        
        mask_up = (neg_log10_padj > fdr_cut) & (logFC_val >= logFC_cut)
        mask_down = (neg_log10_padj > fdr_cut) & (logFC_val <= -logFC_cut)
        mask_ns = ~(mask_up | mask_down)
        
        current_ax.scatter(logFC_val[mask_ns],neg_log10_padj[mask_ns], s = 5,alpha=0.5,
                           label = 'ns',c = 'lightgrey')
        current_ax.scatter(logFC_val[mask_up],neg_log10_padj[mask_up], s = 5,alpha=0.5,
                           label =f'UP (>{logFC_cut})',c = '#D32F2F')
        current_ax.scatter(logFC_val[mask_down],neg_log10_padj[mask_down], s = 5,alpha=0.5,
                           label = f'DOWN (<-{logFC_cut})',c = '#1976D2')
        current_ax.axhline(y=fdr_cut, color='black', linestyle='--', linewidth=0.8)
        current_ax.axvline(x=logFC_cut, color='black', linestyle='--', linewidth=0.8)
        current_ax.axvline(x=-logFC_cut, color='black', linestyle='--', linewidth=0.8)
        current_ax.set_title(f"Cluster: {i}")
        current_ax.set_xlabel("Log2 Fold Change")
        current_ax.set_ylabel("-Log10 Adj. P-value")
        current_ax.legend(loc='upper right', fontsize=8)
        
    plt.tight_layout()
    plt.savefig('/Users/qgem/GitHub/PBMC3k-reproducible/notebooks/figures/volcano.png')
    plt.close()
    #print(color)
    #return neg_log10_padj

In [ ]:
def dot_plot(adata,dot_plot_save):

    # The "Zheng 2017" Marker List
    marker_genes_dict = {
        'B-cells': ['MS4A1', 'CD79A','CD79B'],
        'CD4+ T-cells': ['IL7R', 'CD3D','FHIT','MAL'],
        'CD8+ T-cells': ['CD8A', 'CD8B','GZMK'],
        'NK cells': ['GNLY', 'NKG7','TYROBP'],
        'CD14+ Mono': ['CD14', 'LYZ'],
        'FCGR3A+ Mono': ['FCGR3A', 'MS4A7'],
        'Dendritic': ['FCER1A', 'CST3','HLA-DQA1'],
        'Megakaryocytes': ['PPBP','PF4']
    }
    
    dp = sc.pl.dotplot(adata,marker_genes_dict,groupby='leiden',return_fig=True)
    dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5).savefig(dot_plot_save)
    return dp



In [ ]:
def top_n_genes_dotplot(adata,top_n_genes,rank_dot_plot_save):
    rank_dp = sc.pl.rank_genes_groups_dotplot(adata,n_genes=top_n_genes,
                                              save= rank_dot_plot_save)
    

In [ ]:
def marking_cluster_to_cell_types():
    print('hello')

In [ ]:
def annotating_clusters_final():
    print('hello')

In [ ]:
if __name__ == '__main__':
    adata_B_path = '/Users/qgem/GitHub/PBMC3k-reproducible/data/objects/pbmc3k_clustered_B.h5ad'
    dot_plot_save = '/Users/qgem/GitHub/PBMC3k-reproducible/notebooks/figures/dot_plot.png'
    rank_dot_plot_save = 'rank_dot_plot.png'
    adata = load_evidence(adata_B_path)
    b = adata.obsm['X_umap']
    adata.obsm['spatial'] = adata.obsm['X_umap']
    sc.pl.umap(adata,color='CST3', layer='log1p_norm')
    sc.pl.umap(adata,color='LGALS2', layer='log1p_norm')
    sc.pl.umap(adata,color='S100A8', layer='log1p_norm')
    sc.pl.spatial(adata,color='CST3',spot_size=0.8, layer='log1p_norm')
    '''
    adata= wilcoxon_test(adata)
    #pval_hist(adata)
    #volcano_logfc_pvals_adj(adata)
    #dp = dot_plot(adata,dot_plot_save)
    #top_n_genes_dotplot(adata,3,rank_dot_plot_save)
    keys_g = ['S100A9','CD14','S100A8']
    mask = adata.obs['leiden'] =='5'
    df = adata[mask].to_df()
    ada_arr = np.array(df[keys_g])
    alpha = adata.uns['rank_genes_groups']['names']
    
    for i in alpha.dtype.names:
        top_genes = adata.uns['rank_genes_groups']['names'][i][:3]
        sc.pl.dotplot(adata,top_genes,groupby='leiden',layer='log1p_norm')
    #sc.pl.rank_genes_groups_violin(adata,groups='5',n_genes=3,)
    sc.pl.violin(adata[mask],keys=alpha['5'][:3],groupby='leiden',layer='log1p_norm')
    fig, ax = plt.subplots()
    ax.set_ylabel('expression')
    ax.boxplot(ada_arr)
'''